# ChakraModel Eval Harness — Kaggle Runner (v8)

Verified via a separate CPU-only check before this: v8 discovers 6 datasets totaling 5,539 frames (4 image benchmarks scored individually + 2 video datasets each capped to ~2000 evenly-sampled frames, per-DATASET not per-clip). Expected runtime: well under 30 minutes on T4x2.

Requires:
1. Dataset **ChakraModel_EvalHarness_Code** — the v8 zip — attached
2. Dataset **ChakraModel_EvalHarness_Weights** (chakra_transformer_best.pth + combo1_best.pth) attached
3. Your polyp datasets attached (cvc-clinicdb, etis-larib, kvasir-seg, CVC-300, both Hyperkvasir video parts)
4. Settings: **Accelerator = GPU T4 x2**, **Internet = On**

No patch cells. If anything needs changing, fix the source and re-zip — never edit staged files in a cell.

In [1]:
!pip install -q timm albumentations opencv-python-headless psutil

In [2]:
import os, shutil, glob, sys

WORK = '/kaggle/working'

code_zip = None
for p in glob.glob('/kaggle/input/**/ChakraModel_EvalHarness_Code*.zip', recursive=True):
    code_zip = p
    break

if code_zip:
    print('Unpacking code from', code_zip)
    shutil.unpack_archive(code_zip, WORK)
else:
    eh_dir = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if os.path.basename(root) == 'eval_harness' and 'run_eval.py' in files:
            eh_dir = root
            break
    assert eh_dir, (
        'Could not find ChakraModel_EvalHarness_Code*.zip or an extracted eval_harness/ '
        'folder anywhere under /kaggle/input — attach the v8 code dataset.'
    )
    dataset_root = os.path.dirname(eh_dir)
    print('Found already-extracted code at', dataset_root)
    shutil.copytree(eh_dir, f'{WORK}/eval_harness', dirs_exist_ok=True)
    src_dir = os.path.join(dataset_root, 'src')
    if os.path.isdir(src_dir):
        shutil.copytree(src_dir, f'{WORK}/src', dirs_exist_ok=True)
    print('Copied eval_harness/ and src/ into', WORK)

# --- sanity check: confirm this IS v8 before wasting GPU hours ---
rm_path = f'{WORK}/eval_harness/resource_monitor.py'
ee_path = f'{WORK}/eval_harness/eval_engine.py'
dd_path = f'{WORK}/eval_harness/dataset_discovery.py'
with open(rm_path) as f: rm_src = f.read()
with open(ee_path) as f: ee_src = f.read()
with open(dd_path) as f: dd_src = f.read()
assert 'MAX_BATCH         = 8' in rm_src, 'Stale resource_monitor.py — re-upload the v8 code dataset.'
assert 'mode="default"' in ee_src and '_key_renames' in ee_src, 'Stale eval_engine.py — re-upload the v8 code dataset.'
assert 'MAX_FRAMES_PER_DATASET' in dd_src and '_sampling_plan' in dd_src, 'Stale dataset_discovery.py (not v8) — re-upload the v8 code dataset.'
print('Verified: staged code is v8.')

# Force clean import in case this kernel touched an older copy earlier.
for _m in ['dataset_discovery', 'eval_engine', 'resource_monitor']:
    sys.modules.pop(_m, None)

os.makedirs(f'{WORK}/weights/checkpoints', exist_ok=True)
wanted = {'chakra_transformer_best.pth', 'combo1_best.pth'}
found = set()
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f in wanted and f not in found:
            src = os.path.join(root, f)
            dst = f'{WORK}/weights/checkpoints/{f}'
            shutil.copy(src, dst)
            found.add(f)
            print('Linked', f, 'from', src)
missing = wanted - found
if missing:
    print('WARNING missing weights:', missing, '— attach the ChakraModel_EvalHarness_Weights dataset.')
else:
    print('Both checkpoints staged OK.')

Found already-extracted code at /kaggle/input/datasets/jayasrikannuchamy/chakramodel-evalharness-code-v8
Copied eval_harness/ and src/ into /kaggle/working
Verified: staged code is v8.
Linked combo1_best.pth from /kaggle/input/datasets/jayasrikannuchamy/chakratransformer-weights/weights/checkpoints/combo1_best.pth
Linked chakra_transformer_best.pth from /kaggle/input/datasets/jayasrikannuchamy/chakratransformer-weights/weights/checkpoints/chakra_transformer_best.pth
Both checkpoints staged OK.


In [3]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(' -', torch.cuda.get_device_name(i))
assert torch.cuda.is_available(), 'CUDA GPU not available — select Accelerator = GPU T4 x2 in notebook settings.'

CUDA available: True
GPU count: 2
 - Tesla T4
 - Tesla T4


In [4]:
%cd /kaggle/working
!python eval_harness/run_eval.py

/kaggle/working
[Engine] Detected 2 GPU(s).
[Engine] ViT-Large → cuda:0 | PraNet → cuda:1
[Engine] Multi-GPU parallel mode: True

  ChakraModel Universal Evaluation Harness — FULL STRESS MODE
  PyTorch   : 2.10.0+cu128
  GPUs      : 2
    cuda:0 = Tesla T4  (14.6 GB VRAM)
    cuda:1 = Tesla T4  (14.6 GB VRAM)
  CPU cores : 4
  RAM       : 31.3 GB
  Strategy  : FP16 autocast | CUDA streams | Aggressive batch ramp
  Throttling: NONE — backs off only at >98% VRAM (OOM protection only)

[Config] cudnn.benchmark=True | TF32 matmul=high | deterministic=False

[Main] Dataset root : /kaggle/input

[Main] Scanning for datasets...
  [DISCOVER] cvc-clinicdb                             | PAIRED_IMAGE    |    495 frames
  [DISCOVER] etis-larib                               | PAIRED_IMAGE    |      5 frames
  [DISCOVER] kvasir-seg                               | PAIRED_IMAGE    |    992 frames
  [DISCOVER] CVC-300                                  | PAIRED_IMAGE    |     60 frames
  [DISCOVER] hyper-

In [5]:
# Sanity peek at the results file
import json, glob
res_files = sorted(glob.glob('/kaggle/working/eval_harness/results/evaluation_results_*.json'))
assert res_files, 'No results file produced. Scroll up in the run cell for the traceback.'
with open(res_files[-1]) as f:
    results = json.load(f)
for r in results:
    print(r.get('dataset'), '->', r.get('vit_large', {}).get('dice'), '| PraNet:', r.get('pranet', {}).get('dice'), r.get('error', ''))

cvc-clinicdb -> 0.634468 | PraNet: 0.847594 
etis-larib -> 0.952821 | PraNet: 0.984637 
kvasir-seg -> 0.684354 | PraNet: 0.941002 
CVC-300 -> 0.451103 | PraNet: 0.813916 
hyper-kvasir-labeled-videos-part 2-002 -> 0.0 | PraNet: 0.104588 
hyper-kvasir-labeled-videos_part2-001 -> 0.0 | PraNet: 0.0975 
